[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C22_Reasoning_RL_Course/02_grpo/02_grpo.ipynb)

# 02 · GRPO 从零（用 numpy 模拟）

目标：从零实现 **GRPO（组相对策略优化）**——组内归一化优势、importance ratio、clipped surrogate、KL 正则——在玩具任务上**对拍解析定义**、跑通完整 GRPO 训练，并**正面对比 PPO**（看 GRPO 如何用组采样替代 critic）。

路线：组相对优势(对拍 z-score) → ratio clip(逐情形验证) → KL 估计 → 完整 GRPO update → token vs sequence loss → vs critic 基线 → ✏️ 练习 → 📖 答案 → 🧪 真实组大小/配方胶囊。

> 心智模型：**GRPO = 用「同题一组答案的组内相对好坏」当优势，去掉 critic**。clip 防崩、KL 防偏，是与 PPO 共享的『稳』机制。

## 1 · 组相对优势：减组均值、除组标准差

GRPO 的心脏：$A_i=(r_i-\text{mean})/(\text{std}+\epsilon)$。

我们实现它，对拍 **z-score 的解析定义**，并验证：减均值后**和为 0**（合法基线）、归一化后**均值 0 方差 1**、**全对/全错组优势全 0**（零信号）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def grpo_advantage(rewards, normalize=True, eps=1e-8):
    '''组相对优势：减组均值(基线)，可选除组标准差(归一化)。'''
    r = np.asarray(rewards, dtype=float)
    adv = r - r.mean()
    if normalize:
        adv = adv / (r.std() + eps)
    return adv

r = np.array([1.0, 0.0, 1.0, 0.0, 1.0, 0.0])
adv = grpo_advantage(r)
print('回报   :', r)
print('优势   :', np.round(adv, 4))
# 减均值 -> 和为 0（合法基线，无偏）
assert abs(grpo_advantage(r, normalize=False).sum()) < 1e-9
# 归一化 -> 均值 0、方差 1（对拍 z-score）
assert abs(adv.mean()) < 1e-7 and abs(adv.std() - 1.0) < 1e-6
# 全对组、全错组 -> 优势全 0（零信号，不产生梯度）
assert np.allclose(grpo_advantage(np.ones(4)), 0.0)
assert np.allclose(grpo_advantage(np.zeros(4)), 0.0)
print('✅ 组相对优势：零均值合法基线 + 单位方差 z-score；全对/全错组零梯度')

## 2 · importance ratio 与 clipped surrogate

ratio $\rho=\pi_\theta/\pi_{\text{old}}$ 让我们能用旧 rollout 做多步更新。clipped surrogate $\min(\rho A,\ \text{clip}(\rho,1-\varepsilon,1+\varepsilon)A)$ 限制每步幅度。

逐情形验证 clip 的行为：好动作($A>0$)不过度推高、坏动作($A<0$)不过度压低。

In [ ]:
def clipped_surrogate(ratio, adv, eps=0.2):
    '''PPO/GRPO 的裁剪代理目标(逐元素)。'''
    ratio = np.asarray(ratio, dtype=float)
    adv = np.asarray(adv, dtype=float)
    unclipped = ratio * adv
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * adv
    return np.minimum(unclipped, clipped)

eps = 0.2
# 好动作 A>0：ratio 很大时应被截在 (1+eps)*A
L_big_pos = clipped_surrogate(5.0, 1.0, eps)
assert abs(L_big_pos - (1 + eps) * 1.0) < 1e-9, '好动作 ratio 过大被截上限'
# 坏动作 A<0：ratio 很小时应被截在 (1-eps)*A
L_small_neg = clipped_surrogate(0.1, -1.0, eps)
assert abs(L_small_neg - (1 - eps) * (-1.0)) < 1e-9, '坏动作 ratio 过小被截下限'
# ratio=1（没更新）：surrogate == adv
assert abs(clipped_surrogate(1.0, 0.7, eps) - 0.7) < 1e-9
print('ratio=5,  A=+1 -> surrogate =', round(float(L_big_pos), 3), '(截在 1.2)')
print('ratio=.1, A=-1 -> surrogate =', round(float(L_small_neg), 3), '(截在 -0.8)')
print('✅ clip 行为正确：好动作不过推、坏动作不过压，限制每步信赖域')

## 3 · clip 真的限制了更新幅度吗？

构造一个例子直接证明 clip 的作用：让 ratio 远离 1，对比 clip / 不clip 的目标值，验证 clip **削弱**了过激更新（`min` 保证只会更保守）。

In [ ]:
ratios = np.array([0.1, 0.5, 0.8, 1.0, 1.2, 2.0, 5.0])
for A in [+1.0, -1.0]:
    unclipped = ratios * A
    surrogate = clipped_surrogate(ratios, np.full_like(ratios, A), eps=0.2)
    print(f'A={A:+.0f}: ratio={ratios}')
    print(f'      unclipped={np.round(unclipped,2)}')
    print(f'      surrogate={np.round(surrogate,2)}  (<=unclipped, 更保守)')
    # min 保证 surrogate <= unclipped 当 A>0 区域被裁；总之不会比不裁更激进
    assert np.all(surrogate <= unclipped + 1e-9), 'min 保证不会更激进'
print('✅ clipped surrogate 永不比 unclipped 更激进 —— 这是它防崩的本质')

## 4 · KL 正则：拉住策略别偏离参考

KL$(\pi_\theta\|\pi_{\text{ref}})$ 把策略拴在参考(SFT)附近。精确 KL 需遍历词表；实践用 **k3 估计** $\rho-1-\log\rho$（$\rho=\pi_{\text{ref}}/\pi_\theta$）在采样 token 上近似——无偏且非负、低方差。

我们对拍精确 KL 与 k3 估计。

In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def exact_kl(p, q):
    '''精确 KL(p||q) = sum p log(p/q)。'''
    return float((p * (np.log(p + 1e-12) - np.log(q + 1e-12))).sum())

def k3_kl_estimate(p_theta, p_ref, n=200000, seed=0):
    '''k3 估计：在 pi_theta 上采 token，平均 (r - 1 - log r), r = p_ref/p_theta。
       它是 KL(pi_theta || pi_ref) 的无偏、非负、低方差估计。'''
    r = np.random.default_rng(seed)
    samples = r.choice(len(p_theta), size=n, p=p_theta)
    ratio = p_ref[samples] / p_theta[samples]
    return float(np.mean(ratio - 1 - np.log(ratio)))

theta = softmax(np.array([1.0, 0.5, 0.0, -0.5, 0.2]))
ref   = softmax(np.array([0.8, 0.6, 0.1, -0.3, 0.0]))
kl_exact = exact_kl(theta, ref)
kl_k3 = k3_kl_estimate(theta, ref)
print(f'精确 KL(theta||ref) = {kl_exact:.5f}')
print(f'k3   估计           = {kl_k3:.5f}')
assert abs(kl_exact - kl_k3) < 0.01, 'k3 估计应逼近精确 KL'
assert kl_k3 >= 0, 'KL 非负'
# theta==ref 时 KL=0
assert abs(exact_kl(theta, theta)) < 1e-9
print('✅ k3 估计对拍精确 KL：实践中用它在采样 token 上近似 KL 正则')

## 5 · 完整 GRPO update step

把组优势 + ratio clip + KL 拼成一次完整 GRPO 更新，在凑数玩具任务上训练，验证**成功率单调上升**。
这就是 trl `GRPOTrainer` 的玩具内核。

In [ ]:
from itertools import product
ACTIONS = np.array([0, 1, 2, 3]); N_STEPS = 3; TARGET = 6; AA = len(ACTIONS)

def step_softmax(logits, temp=1.0):
    z = logits / temp; z = z - z.max(axis=-1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)
def reward(traj): return float(sum(ACTIONS[a] for a in traj) == TARGET)
def exact_success(logits, temp=1.0):
    P = step_softmax(logits, temp); J = 0.0
    for tr in product(range(AA), repeat=N_STEPS):
        pt = 1.0
        for t, a in enumerate(tr): pt *= P[t][a]
        J += pt * reward(tr)
    return J
def traj_logprob(logits, traj, temp=1.0):
    P = step_softmax(logits, temp)
    return sum(np.log(P[t][a] + 1e-12) for t, a in enumerate(traj))
def dlogp(logits, traj, temp=1.0):
    P = step_softmax(logits, temp); g = np.zeros_like(logits)
    for t, a in enumerate(traj):
        oh = np.zeros(AA); oh[a] = 1.0; g[t] += (oh - P[t]) / temp
    return g

def grpo_train(steps=150, group=128, lr=1.0, eps=0.2, inner=2, temp=1.0, seed=0):
    r = np.random.default_rng(seed)
    logits = np.zeros((N_STEPS, AA))
    hist = [exact_success(logits, temp)]
    for _ in range(steps):
        old_logits = logits.copy()                      # 旧策略快照
        P = step_softmax(old_logits, temp)
        trajs = [[r.choice(AA, p=P[t]) for t in range(N_STEPS)] for _ in range(group)]
        Rs = np.array([reward(tr) for tr in trajs])
        adv = grpo_advantage(Rs, normalize=True)        # 组相对优势
        old_lp = np.array([traj_logprob(old_logits, tr, temp) for tr in trajs])
        for _ in range(inner):                          # 用同批 rollout 多步更新
            grad = np.zeros_like(logits)
            for tr, A_i, olp in zip(trajs, adv, old_lp):
                new_lp = traj_logprob(logits, tr, temp)
                ratio = np.exp(new_lp - olp)
                # clipped surrogate 的梯度：未裁剪时 = ratio*A*dlogp，裁剪区梯度为 0
                if (A_i >= 0 and ratio <= 1 + eps) or (A_i < 0 and ratio >= 1 - eps):
                    grad += ratio * A_i * dlogp(logits, tr, temp)
            grad /= group
            logits = logits + lr * grad
        hist.append(exact_success(logits, temp))
    return hist

hist = grpo_train()
print(f'GRPO 训练: 成功率 {hist[0]:.3f} -> {hist[-1]:.3f}')
print('轨迹(每25步):', [f'{h:.2f}' for h in hist[::25]])
assert hist[-1] > hist[0] + 0.3 and hist[-1] > 0.8, 'GRPO 应把成功率训上去'
print('✅ 完整 GRPO（组优势+ratio clip+多步更新）把成功率训到接近最优')

## 6 · token-level vs sequence-level loss

GRPO 原版按**序列**平均(每条先内部平均)，长序列每 token 权重被稀释；DAPO 改 **token-level**(所有 token 同权)。
我们用不等长回答构造例子，看两种聚合给同一个 token 的权重差异。

In [ ]:
def seq_level_loss(per_token, lengths):
    '''每条先按自身长度平均，再跨条平均(序列级)。per_token: list of arrays。'''
    seq = [pt.mean() for pt in per_token]
    return float(np.mean(seq))

def token_level_loss(per_token, lengths):
    '''所有 token 拼一起，跨整组求和再除总 token 数(token 级)。'''
    allt = np.concatenate(per_token)
    return float(allt.sum() / allt.size)

# 两条回答：短(2 token)全 1，长(8 token)全 1
short = np.ones(2); long = np.ones(8)
pt = [short, long]; lens = [2, 8]
sl = seq_level_loss(pt, lens); tl = token_level_loss(pt, lens)
print(f'序列级 loss = {sl:.4f}  (短长回答各占 1/2，与长度无关)')
print(f'token 级 loss = {tl:.4f}  (每 token 同权)')
# 全 1 时两者都为 1；差异在权重分配。构造一个有区分的例子：
short2 = np.full(2, 2.0); long2 = np.zeros(8)
pt2 = [short2, long2]
sl2 = seq_level_loss(pt2, [2, 8]); tl2 = token_level_loss(pt2, [2, 8])
print(f'\n短回答信号强(2.0)、长回答信号0:')
print(f'  序列级={sl2:.4f} (短回答占 1/2 权重 -> 1.0)')
print(f'  token级={tl2:.4f} (2 个强 token / 10 个 -> 0.4)')
assert abs(sl2 - 1.0) < 1e-9 and abs(tl2 - 0.4) < 1e-9
assert sl2 != tl2, '两种聚合对不等长回答给出不同权重'
print('✅ token-level 让长回答的每个 token 不被稀释 —— DAPO 在长 CoT 上的关键改动')

## 7 · 组均值基线 ≈ critic 的作用

GRPO 用组均值替代 critic。验证：**组均值是该 prompt 期望回报 $E[R|q]$ 的无偏估计**——即 critic 想学的那个 $V(q)$。这就是「为什么去掉 critic 还能工作」。

In [ ]:
# 一个 prompt 的真实期望回报(用大量采样的均值当 ground truth = critic 的目标 V(q))
logits_fixed = np.array([[0.5, 0.2, 0.0, -0.3]] * N_STEPS)
P = step_softmax(logits_fixed)
true_V = exact_success(logits_fixed)         # 精确 E[R|q]
print(f'真实 V(q) = E[R|q] = {true_V:.4f}  (critic 想学的目标)')

# GRPO 的组均值(小组)是 V(q) 的无偏估计；组越大越准
for G in [4, 16, 64, 256]:
    means = []
    for _ in range(500):
        trajs = [[rng.choice(AA, p=P[t]) for t in range(N_STEPS)] for _ in range(G)]
        means.append(np.mean([reward(tr) for tr in trajs]))
    means = np.array(means)
    print(f'  组大小 G={G:3d}: 组均值 期望={means.mean():.4f}  标准差={means.std():.4f}')
    assert abs(means.mean() - true_V) < 0.03, '组均值应是 V(q) 的无偏估计'
print('✅ 组均值 = E[R|q] 的无偏估计(critic 的目标)；G 越大方差越小 -> 去 critic 合理')

---
## ✏️ 练习 1：组相对优势（含只减均值变体）

实现 `grpo_advantage(rewards, normalize, eps)`。验证两种模式的性质，并说明全对组为何零梯度。

额外：实现 `is_zero_signal_group(rewards)` 判断一个组是否全对或全错（应被 dynamic sampling 过滤）。

In [ ]:
def grpo_advantage(rewards, normalize=True, eps=1e-8):
    # TODO: adv = r - mean(r); if normalize: adv /= (std(r)+eps)
    raise NotImplementedError

def is_zero_signal_group(rewards):
    # TODO: 全相同(全对/全错)即零信号 -> True
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r = np.array([1.0, 0.0, 0.0, 1.0])
assert abs(grpo_advantage(r, normalize=False).sum()) < 1e-9
a = grpo_advantage(r, normalize=True)
assert abs(a.mean()) < 1e-7 and abs(a.std() - 1) < 1e-6
assert np.allclose(grpo_advantage(np.ones(5)), 0.0)
assert is_zero_signal_group(np.ones(4)) and is_zero_signal_group(np.zeros(4))
assert not is_zero_signal_group(np.array([1.0, 0.0, 1.0]))
print('✅ 练习 1 通过：组优势 + 零信号组检测(dynamic sampling 的基础)')

## ✏️ 练习 2：clipped surrogate 与触发判定

实现 `clipped_surrogate(ratio, adv, eps)` 和 `is_clipped(ratio, adv, eps)`(该样本是否落在裁剪区、梯度为 0)。

In [ ]:
def clipped_surrogate(ratio, adv, eps=0.2):
    # TODO: min(ratio*adv, clip(ratio,1-eps,1+eps)*adv)
    raise NotImplementedError

def is_clipped(ratio, adv, eps=0.2):
    # TODO: True 当裁剪项 < 未裁剪项(即 min 取了裁剪项 -> 该方向梯度被截为 0)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert abs(clipped_surrogate(5.0, 1.0) - 1.2) < 1e-9
assert abs(clipped_surrogate(0.1, -1.0) - (-0.8)) < 1e-9
assert abs(clipped_surrogate(1.0, 0.5) - 0.5) < 1e-9
# 好动作 ratio 过大 -> 被裁
assert is_clipped(5.0, 1.0) == True
# ratio 在范围内 -> 不裁
assert is_clipped(1.1, 1.0) == False
# 坏动作 ratio 过小 -> 被裁
assert is_clipped(0.1, -1.0) == True
print('✅ 练习 2 通过：裁剪目标 + 裁剪触发判定')

## ✏️ 练习 3：k3 KL 估计

实现 `k3_kl(p_theta, p_ref, n, seed)`：在 `p_theta` 上采样、用 $\rho-1-\log\rho$（$\rho=p_{\text{ref}}/p_{\theta}$）估计 KL$(\theta\|\text{ref})$。验证它逼近精确 KL、非负、且 $\theta=\text{ref}$ 时为 0。

In [ ]:
def k3_kl(p_theta, p_ref, n=200000, seed=0):
    # TODO: 采样 token, ratio = p_ref[s]/p_theta[s], 返回 mean(ratio - 1 - log ratio)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()
def exact_kl(p, q):
    return float((p * (np.log(p+1e-12) - np.log(q+1e-12))).sum())
pt = softmax(np.array([1.0, 0.0, -0.5, 0.3]))
pr = softmax(np.array([0.7, 0.1, -0.2, 0.0]))
assert abs(k3_kl(pt, pr) - exact_kl(pt, pr)) < 0.01, 'k3 应逼近精确 KL'
assert k3_kl(pt, pr) >= 0
assert k3_kl(pt, pt) < 1e-6, 'theta==ref 时 KL=0'
print('✅ 练习 3 通过：k3 KL 估计(实践中的 KL 正则估计法)')

## ✏️ 练习 4：PPO vs GRPO —— 谁更省、基线从哪来

实现 `count_models(algo)` 返回该算法需同时驻留的大模型数(策略+critic+ref+old；old 可与策略共享参数不另计)；并实现 `baseline_value(algo, group_rewards, critic_value)` 返回各自的基线。

In [ ]:
def count_models(algo):
    # TODO: 返回需要的『独立大模型』数量
    #   PPO : 策略 + critic + 参考 = 3（old 是策略快照不另计）
    #   GRPO: 策略 + 参考 = 2（无 critic）
    raise NotImplementedError

def baseline_value(algo, group_rewards, critic_value=None):
    # TODO: PPO 用 critic_value；GRPO 用 group_rewards 的均值
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert count_models('PPO') == 3
assert count_models('GRPO') == 2
assert count_models('GRPO') < count_models('PPO'), 'GRPO 省一个 critic'
g = np.array([1.0, 0.0, 1.0, 0.0])
assert abs(baseline_value('GRPO', g) - 0.5) < 1e-9, 'GRPO 基线 = 组均值'
assert abs(baseline_value('PPO', g, critic_value=0.42) - 0.42) < 1e-9, 'PPO 基线 = critic'
print('✅ 练习 4 通过：GRPO 用组均值替代 critic，省一个大模型')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def grpo_advantage(rewards, normalize=True, eps=1e-8):
    r = np.asarray(rewards, dtype=float)
    adv = r - r.mean()
    if normalize:
        adv = adv / (r.std() + eps)
    return adv

def is_zero_signal_group(rewards):
    r = np.asarray(rewards, dtype=float)
    return bool(np.all(r == r[0]))

In [ ]:
# 练习 2 参考答案
def clipped_surrogate(ratio, adv, eps=0.2):
    ratio = np.asarray(ratio, dtype=float); adv = np.asarray(adv, dtype=float)
    return np.minimum(ratio * adv, np.clip(ratio, 1 - eps, 1 + eps) * adv)

def is_clipped(ratio, adv, eps=0.2):
    unclipped = ratio * adv
    clipped = np.clip(ratio, 1 - eps, 1 + eps) * adv
    return bool(clipped < unclipped - 1e-12)

In [ ]:
# 练习 3 参考答案
def k3_kl(p_theta, p_ref, n=200000, seed=0):
    r = np.random.default_rng(seed)
    s = r.choice(len(p_theta), size=n, p=p_theta)
    ratio = p_ref[s] / p_theta[s]
    return float(np.mean(ratio - 1 - np.log(ratio)))

In [ ]:
# 练习 4 参考答案
def count_models(algo):
    return {'PPO': 3, 'GRPO': 2}[algo]

def baseline_value(algo, group_rewards, critic_value=None):
    if algo == 'GRPO':
        return float(np.mean(group_rewards))
    return float(critic_value)

---
## 🧪 真实数据胶囊：GRPO 的真实配方与显存账

用 DeepSeek-R1 / DeepSeekMath / DAPO 报告里的**真实超参**，算一笔 GRPO 的账：
① 去掉 critic 省多少显存(策略与 critic 同规模)；② 真实组大小 $G$ 下，全对/全错组的浪费比例。

In [ ]:
# 真实 GRPO 配方(取自公开报告，约数)
GRPO_CONFIGS = {
    'DeepSeekMath-GRPO': dict(group_G=64,  clip_eps=0.2,  kl_beta=0.04),
    'DeepSeek-R1(风格)': dict(group_G=64,  clip_eps=0.2,  kl_beta=0.001),  # KL 极小
    'DAPO(风格)':        dict(group_G=512, clip_eps=0.2,  kl_beta=0.0),    # 去 KL + clip-higher
}
for name, cfg in GRPO_CONFIGS.items():
    print(f'{name:20s} G={cfg["group_G"]:4d}  clip={cfg["clip_eps"]}  KL_beta={cfg["kl_beta"]}')

# 显存账：critic 与策略同规模 -> 去 critic 省下约 策略权重+优化器状态 的份额
def memory_saving_no_critic(policy_params_B, bytes_per_param=2, optimizer_mult=3):
    '''去掉 critic 省的显存(GB)：critic 权重 + 其优化器状态(Adam ~2x) + 梯度。
       近似 = policy 同规模的 (1 权重 + optimizer_mult 状态) * bytes。'''
    weights = policy_params_B * 1e9 * bytes_per_param
    total = weights * (1 + optimizer_mult)      # 权重 + 优化器/梯度状态
    return total / 1e9

for B in [7, 32, 70]:
    saved = memory_saving_no_critic(B)
    print(f'  {B}B 模型: 去 critic 约省 {saved:.0f} GB 显存(权重+优化器状态)')
assert memory_saving_no_critic(70) > memory_saving_no_critic(7)
print('✅ 去 critic 在大模型上省下可观显存 —— GRPO 的核心工程收益之一')

**🧪 胶囊练习**：实现 `wasted_fraction(p_correct, G, n_prompts, seed)` —— 在单题正确率 `p_correct`、组大小 `G` 下，**全对或全错(零梯度)组占的比例**(用二项分布模拟)。它量化 dynamic sampling 能省下多少浪费。

In [ ]:
def wasted_fraction(p_correct, G, n_prompts=20000, seed=0):
    # TODO: 对 n_prompts 个组各采 G 个 Bernoulli(p_correct)，
    #       统计『全对(全1)或全错(全0)』的组占比
    raise NotImplementedError

In [ ]:
# 自测
r = np.random.default_rng(0)
# p=0.5, G=4: 全对 (.5)^4 + 全错 (.5)^4 = 2/16 = 0.125
w = wasted_fraction(0.5, 4)
assert abs(w - 0.125) < 0.02, '理论浪费 ~12.5%'
# 极易(p=0.95)或极难(p=0.05)的题浪费更多(常全对/全错)
assert wasted_fraction(0.95, 4) > wasted_fraction(0.5, 4)
# 组越大越不容易全对/全错
assert wasted_fraction(0.5, 16) < wasted_fraction(0.5, 4)
print(f'p=0.5,G=4 浪费={w:.3f}; 这就是 dynamic sampling 要回收的零梯度组')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def wasted_fraction(p_correct, G, n_prompts=20000, seed=0):
    r = np.random.default_rng(seed)
    correct_counts = r.binomial(G, p_correct, size=n_prompts)
    wasted = (correct_counts == 0) | (correct_counts == G)
    return float(np.mean(wasted))

### 小结
- **GRPO = 用组内归一化回报当优势，去掉 critic**。组均值是 $E[R|q]$ 的无偏估计(critic 的目标)，用计算(多采样)换显存与稳定。
- **组相对优势** $A_i=(r_i-\text{mean})/(\text{std}+\epsilon)$：零均值合法基线 + 单位方差；**全对/全错组零梯度**(dynamic sampling 要过滤)。
- **ratio clip**(与 PPO 共享)：$\min(\rho A,\text{clip}(\rho,1\pm\varepsilon)A)$ 限制每步信赖域，好动作不过推、坏动作不过压。
- **KL 正则**(k3 估计)：拉住策略别偏离参考；可验证域可调极小(奖励难 hack)，学习奖励域须拉紧(防过度优化)。
- **token-level loss**(DAPO)：避免长 CoT 的 token 被序列级平均稀释。
- **GRPO vs PPO**：只差『优势怎么来』，却牵动显存/稳定/超参——约束变了，最优解就变了。

下一站：**模块 03 · 过程奖励 PRM** —— 从『奖励的算法』转向『奖励的粒度』，给推理的每一步打分。